In [ ]:
# Импортируйте библиотеку pandas как pd. Загрузите датасет bookings.csv с разделителем ;. Проверьте размер таблицы, типы переменных, а затем выведите первые 7 строк, чтобы посмотреть на данные.

In [1]:
import pandas as pd

In [2]:
df = pd.read_csv('C:/Work/Analytics/2_bookings.csv', encoding = 'windows-1251', sep = ';')

In [4]:
df.head(7)

,Hotel,Is Canceled,Lead Time,arrival full date,Arrival Date Year,Arrival Date Month,Arrival Date Week Number,Arrival Date Day of Month,Stays in Weekend nights,Stays in week nights,...,Adults,Children,Babies,Meal,Country,Reserved Room Type,Assigned room type,customer type,Reservation Status,Reservation status_date
0,Resort Hotel,0,342,2015-07-01,2015,July,27,1,0,0,...,2,0.0,0,BB,PRT,C,C,Transient,Check-Out,2015-07-01
1,Resort Hotel,0,737,2015-07-01,2015,July,27,1,0,0,...,2,0.0,0,BB,PRT,C,C,Transient,Check-Out,2015-07-01
2,Resort Hotel,0,7,2015-07-01,2015,July,27,1,0,1,...,1,0.0,0,BB,GBR,A,C,Transient,Check-Out,2015-07-02
3,Resort Hotel,0,13,2015-07-01,2015,July,27,1,0,1,...,1,0.0,0,BB,GBR,A,A,Transient,Check-Out,2015-07-02
4,Resort Hotel,0,14,2015-07-01,2015,July,27,1,0,2,...,2,0.0,0,BB,GBR,A,A,Transient,Check-Out,2015-07-03
5,Resort Hotel,0,14,2015-07-01,2015,July,27,1,0,2,...,2,0.0,0,BB,GBR,A,A,Transient,Check-Out,2015-07-03
6,Resort Hotel,0,0,2015-07-01,2015,July,27,1,0,2,...,2,0.0,0,BB,PRT,C,C,Transient,Check-Out,2015-07-03


In [5]:
df.columns

Index(['Hotel', 'Is Canceled', 'Lead Time', 'arrival full date',
       'Arrival Date Year', 'Arrival Date Month', 'Arrival Date Week Number',
       'Arrival Date Day of Month', 'Stays in Weekend nights',
       'Stays in week nights', 'stays total nights', 'Adults', 'Children',
       'Babies', 'Meal', 'Country', 'Reserved Room Type', 'Assigned room type',
       'customer type', 'Reservation Status', 'Reservation status_date'],
      dtype='str')

In [6]:
def new_column(column):
    new_columns = []
    for i in column:
        new_columns.append(i.lower().replace(' ', '_'))
    return new_columns
df.columns = new_column(df.columns)
df.columns

Index(['hotel', 'is_canceled', 'lead_time', 'arrival_full_date',
       'arrival_date_year', 'arrival_date_month', 'arrival_date_week_number',
       'arrival_date_day_of_month', 'stays_in_weekend_nights',
       'stays_in_week_nights', 'stays_total_nights', 'adults', 'children',
       'babies', 'meal', 'country', 'reserved_room_type', 'assigned_room_type',
       'customer_type', 'reservation_status', 'reservation_status_date'],
      dtype='str')

In [17]:
# Пользователи из каких стран совершили наибольшее число успешных бронирований? Укажите топ-5.
bookings_countries = df.query("reservation_status == 'Check-Out'") \
    .groupby('country', as_index = False)['reservation_status'].count() \
    .sort_values('reservation_status', ascending = False).head(5)
bookings_countries

,country,reservation_status
125,PRT,21071
57,GBR,9676
54,FRA,8481
50,ESP,6391
42,DEU,6069


In [18]:
#На сколько ночей в среднем бронируют отели разных типов?
nights_avg = df.groupby('hotel', as_index = False) \
    .agg({'stays_total_nights': 'mean'})
nights_avg

,hotel,stays_total_nights
0,City Hotel,2.978142
1,Resort Hotel,4.318547


In [9]:
# Иногда тип номера, полученного клиентом (assigned_room_type), отличается от изначально забронированного (reserved_room_type). Такое может произойти, например, по причине овербукинга. Сколько подобных наблюдений встретилось в датасете?

assigned_reserved = df.query("assigned_room_type != reserved_room_type").shape[0]
assigned_reserved


14917

In [19]:
# Проанализируйте даты запланированного прибытия.
# – На какой месяц чаще всего успешно оформляли бронь в 2016?
#  Изменился ли самый популярный месяц в 2017?
# – Сгруппируйте данные по годам и проверьте, на какой месяц бронирования отеля типа City Hotel отменялись чаще всего в каждый из периодов

successful_2016 = df.query("reservation_status != 'Canceled' and arrival_date_year == 2016").groupby('arrival_date_month').size() \
    .sort_values(ascending = False)
successful_2016

arrival_date_month
October      3744
May          3628
April        3421
March        3418
September    3401
August       3277
June         3246
July         3111
November     2857
February     2699
December     2495
January      1741
dtype: int64

In [20]:
# или если смотреть на все годы
successful_2016_n = (
    df.query("reservation_status != 'Canceled'")
    .groupby(['arrival_date_year', 'arrival_date_month'])
    .agg({'hotel': 'count'})
    .rename(columns={'hotel': 'count'})
    .sort_values(['arrival_date_year', 'count'] ,ascending = False)
                     )
successful_2016_n

count
arrival_date_year arrival_date_month       
2017              May                  3618
                  July                 3360
                  March                3349
                  June                 3242
                  April                3221
                  August               3142
                  February             2902
                  January              2456
2016              October              3744
                  May                  3628
                  April                3421
                  March                3418
                  September            3401
                  August               3277
                  June                 3246
                  July                 3111
                  November             2857
                  February             2699
                  December             2495
                  January              1741
2015              October              3243
                  September            3058
                  August               2311
                  December             1996
                  November             1893
                  July                 1544

In [21]:
# на какой месяц бронирования отеля типа City Hotel отменялись чаще всего в каждый из периодов

no_successful = (
    df.query("is_canceled == 1 and hotel == 'City Hotel'")
    .groupby(['arrival_date_year', 'arrival_date_month'])
    .agg({'hotel': 'count'})
    .rename(columns={'hotel': 'count'})
    .sort_values(['arrival_date_year', 'count'] ,ascending = False)
                     )
no_successful

count
arrival_date_year arrival_date_month       
2017              May                  2217
                  April                1926
                  June                 1808
                  July                 1324
                  March                1278
                  August               1123
                  January              1044
                  February              971
2016              October              1947
                  June                 1720
                  September            1567
                  April                1539
                  May                  1436
                  November             1360
                  August               1247
                  March                1108
                  December             1072
                  July                 1043
                  February              930
                  January               438
2015              September            1543
                  October              1321
                  August               1232
                  July                  939
                  December              668
                  November              301

In [22]:
no_success = df.query("is_canceled == 1 and hotel == 'City Hotel'").groupby('arrival_date_year')['arrival_date_month'].value_counts()
no_success

arrival_date_year  arrival_date_month
2015               September             1543
                   October               1321
                   August                1232
                   July                   939
                   December               668
                   November               301
2016               October               1947
                   June                  1720
                   September             1567
                   April                 1539
                   May                   1436
                   November              1360
                   August                1247
                   March                 1108
                   December              1072
                   July                  1043
                   February               930
                   January                438
2017               May                   2217
                   April                 1926
                   June                  1

In [23]:
# Посмотрите на числовые характеристики трёх переменных: adults, children и babies. Какая из них имеет наибольшее среднее значение?
human_count = df[['babies', 'adults', 'children']].mean()
human_count

babies      0.007949
adults      1.856403
children    0.103890
dtype: float64

In [15]:
# Создайте колонку total_kids, объединив children и babies. Для отелей какого типа среднее значение переменной оказалось наибольшим?

df['total_kids'] = df['babies'] + df['children']
result = df.groupby('hotel')['total_kids'].mean()

result

hotel
City Hotel      0.096311
Resort Hotel    0.142586
Name: total_kids, dtype: float64

In [16]:
# Создайте переменную has_kids, которая принимает значение True, если клиент при бронировании указал хотя бы одного ребенка (total_kids), в противном случае – False. Посчитайте отношение количества ушедших пользователей к общему количеству клиентов, выраженное в процентах (churn rate). Укажите, среди какой группы показатель выше.

df['has_kids'] = df['total_kids'] > 0
churn_rate = df.groupby('has_kids')['is_canceled'].mean() * 100
churn_rate

has_kids
False    37.221283
True     34.922846
Name: is_canceled, dtype: float64